In [1]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims
import numpy as np

ModuleNotFoundError: No module named 'anatomist'

In [3]:
SUBJECT = 394956
SIDE = "R"
mni_icbm152_path = "/casa/host/src/brainvisa-share/master/anatomical_templates/mni_icbm152_nlin_asym_09c.nii.gz"
skeleton_path = f"/neurospin/dico/data/deep_folding/current/datasets/hcp/skeletons/2mm/{SIDE}/{SIDE}resampled_skeleton_{SUBJECT}.nii.gz"
white_mesh_path = f"/neurospin/dico/data/bv_databases/human/not_labeled/hcp/hcp/{SUBJECT}/t1mri/BL/default_analysis/segmentation/mesh/{SUBJECT}_{SIDE}white.gii"
grey_mesh_path = f"/neurospin/dico/data/bv_databases/human/not_labeled/hcp/hcp/{SUBJECT}/t1mri/BL/default_analysis/segmentation/mesh/{SUBJECT}_{SIDE}hemi.gii"

In [4]:
def to_bucket(obj):
    if obj.type() == obj.BUCKET:
        return obj
    avol = a.toAimsObject(obj)
    c = aims.Converter(intype=avol, outtype=aims.BucketMap_VOID)
    abck = c(avol)
    bck = a.toAObject(abck)
    bck.releaseAppRef()
    return bck

### To load the white mesh of the subject

In [6]:
window1 = a.createWindow("3D")
white_mesh_obj = a.loadObject(white_mesh_path)
white_mesh_obj.loadReferentialFromHeader()
window1.addObjects(white_mesh_obj)

nifti transfo: 2


In [8]:
window2 = a.createWindow("Axial")
skeleton_obj = a.loadObject(skeleton_path)
skeleton_obj.loadReferentialFromHeader()
window2.addObjects(skeleton_obj)

nifti transfo: 1


ATransformSet::unregisterObserver: ref 0x626057ac7110 not found


In [9]:
window3 = a.createWindow("3D")
grey_mesh_obj = a.loadObject(grey_mesh_path)
grey_mesh_obj.loadReferentialFromHeader()
window3.addObjects(grey_mesh_obj)

nifti transfo: 2


In [10]:
window4 = a.createWindow("Axial")
mni_icbm152_obj = a.loadObject(mni_icbm152_path)
mni_icbm152_obj.loadReferentialFromHeader()
window4.addObjects(mni_icbm152_obj)

nifti transfo: 1


### To see what's in the skeleton header

In [11]:
skeleton = aims.read(skeleton_path)
skeleton_hdr = skeleton.header()
#for key in skeleton_hdr.keys():
#    print(key)
list(skeleton_hdr['referentials'])

['Scanner-based anatomical coordinates']

### What's in the icbm152 template header

In [12]:
icbm152_template = aims.read(mni_icbm152_path)
list(icbm152_template.header()['referentials'])

['Talairach-MNI template-SPM']

### To see what's in the white mesh header

In [13]:
white_mesh = aims.read(white_mesh_path)
print('Keys in white mesh file header:')
#for key in white_mesh.header().keys():
#    print(key)
print('\n','referentials')
print(white_mesh.header()['referentials'])
print('\n','referential')
print(white_mesh.header()['referential'])
print('\n','transformations')
print(white_mesh.header()['transformations'])

Keys in white mesh file header:

 referentials
["Scanner-based anatomical coordinates", "Talairach-MNI template-SPM"]

 referential
b4294ba9-3e81-2402-b4a8-3b5a46ebc00c

 transformations
[ [ -1, 0, 0, 91, 0, -1, 0, 108.151885986328, 0, 0, -1, 90.3015747070312, 0, 0, 0, 1 ], [ -1, 0, 0, 90, 0, -0.999982595443726, 0, 91, 0, 0, -0.999982595443726, 109.300003051758, 0, 0, 0, 1 ] ]


### To apply the transformations to get to the mni icbm152 referential

In [30]:
#dir(aims.AffineTransformation3d())

In [14]:
Traw_mni = aims.AffineTransformation3d(white_mesh.header()['transformations'][1])
Traw_scanner = aims.AffineTransformation3d(skeleton_hdr['transformations'][0])
T_skel_to_mni = Traw_mni #* Traw_scanner.inverse()
print(T_skel_to_mni)

[[ -1.          0.          0.         90.       ]
 [  0.         -0.9999826   0.         91.       ]
 [  0.          0.         -0.9999826 109.3      ]
 [  0.          0.          0.          1.       ]]


In [15]:
vx, vy, vz = skeleton_hdr['voxel_size'][:3]

translation_origine = aims.AffineTransformation3d()
translation_origine.setTranslation([
    0.25 * vx,
    0.25 * vy,
    0.25 * vz
])

translation_origine.setTranslation([0, 0, 0])
translation_origine

[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]

In [16]:
T_final = translation_origine * T_skel_to_mni
skeleton_hdr['transformations'].append(T_final.toVector())
skeleton_hdr['referentials'].append('Talairach-MNI')

In [ ]:
skeleton_rec_obj = a.toAObject(skeleton)
skeleton_rec_obj.loadReferentialFromHeader()
skeleton_rec_bkt = to_bucket(skeleton_rec_obj)
window1.addObjects(skeleton_rec_bkt)